# 04 — Advanced RAG Components

**Phase 5:** NER · Embeddings · HyDE · Query Rewriting · Hybrid Search

All components work offline (no API key needed — regex/synonym fallbacks).  
Set `GROQ_API_KEY` to activate LLM-powered NER, HyDE, and query rewriting.

In [ ]:
import sys, os, json
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.modules.ner_extractor  import NERExtractor
from src.utils.embedder         import Embedder
from src.utils.query_optimizer  import QueryOptimizer
from src.utils.hybrid_search    import HybridSearch

plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})
api_key = os.getenv('GROQ_API_KEY','')
print(f'GROQ_API_KEY: {"set ✓" if api_key else "not set — fallbacks active"}')

## 1. NER — Named Entity Recognition

In [ ]:
ner = NERExtractor()
print(ner)

NER_TESTS = [
    "I've been feeling anxious about work for 3 weeks",
    "I can't sleep and feel completely hopeless since last month",
    "My panic attacks are unbearable, especially in social situations",
    "Feeling a little sad and stressed about school exams sometimes",
    "I've been grieving the loss of my relationship for a few months",
    "Struggling with depression and burnout due to family pressure",
]

print(f'\n{"Text":<55}  {"Symptoms":<20}  {"Triggers":<18}  {"Duration":<14}  Sev')
print('-'*120)
ner_results = []
for text in NER_TESTS:
    r = ner.extract(text)
    ner_results.append({'text': text, **r})
    sym  = ', '.join(r['symptoms'][:2]) or '—'
    trig = ', '.join(r['triggers'][:2]) or '—'
    dur  = r['duration'] or '—'
    print(f"  {text[:53]:<55}  {sym:<20}  {trig:<18}  {dur:<14}  {r['severity']}")

In [ ]:
# Visualise entity frequency across test cases
all_symptoms = [s for r in ner_results for s in r['symptoms']]
all_triggers = [t for r in ner_results for t in r['triggers']]
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Symptoms
if all_symptoms:
    sc = Counter(all_symptoms).most_common(8)
    axes[0].barh([x[0] for x in sc], [x[1] for x in sc], color='#e74c3c')
axes[0].set_title('Top Symptoms Extracted', fontsize=11)

# Triggers
if all_triggers:
    tc = Counter(all_triggers).most_common(8)
    axes[1].barh([x[0] for x in tc], [x[1] for x in tc], color='#3498db')
axes[1].set_title('Top Triggers Extracted', fontsize=11)

# Severity distribution
sev_counts = Counter(r['severity'] for r in ner_results)
sev_colors = {'high':'#e74c3c','medium':'#f39c12','low':'#2ecc71'}
axes[2].bar(sev_counts.keys(), sev_counts.values(),
            color=[sev_colors.get(k,'grey') for k in sev_counts])
axes[2].set_title('Severity Distribution', fontsize=11)

plt.suptitle('NER Extraction Results', fontsize=13)
plt.tight_layout(); plt.show()

## 2. Embedding Pipeline

In [ ]:
emb = Embedder()
print(emb)

# Single embed
v = emb.embed_text('I feel anxious about my job interview')
print(f'\nembed_text: shape={v.shape}  dtype={v.dtype}  norm={np.linalg.norm(v):.5f}')

# Batch
texts = [
    'Anxiety coping strategies and breathing exercises',
    'Depression treatment options including therapy',
    'Mindfulness meditation for stress reduction',
    'Sleep disorders and insomnia management',
    ''  # empty string — should give zero vector
]
vecs = emb.embed_batch(texts)
print(f'embed_batch: shape={vecs.shape}')
print(f'  empty string → zero vector: {np.all(vecs[-1] == 0)}')
for i, t in enumerate(texts[:-1]):
    print(f'  [{i}] norm={np.linalg.norm(vecs[i]):.4f}  {t[:45]}')

In [ ]:
# Semantic similarity matrix
test_pairs = [
    ('I feel anxious',            'I am worried and nervous'),
    ('I feel anxious',            'I feel depressed'),
    ('I feel anxious',            'The weather is nice today'),
    ('depression treatment',      'therapy for sadness'),
    ('panic attack symptoms',     'anxiety physical symptoms'),
]
print(f'{"Pair":<60}  Similarity')
print('-'*75)
for a, b in test_pairs:
    va, vb = emb.embed_text(a), emb.embed_text(b)
    sim = emb.cosine_similarity(va, vb)
    bar = '█' * int(sim * 20)
    print(f"  {a[:28]!r:<30} ↔ {b[:28]!r:<30}  {sim:.3f}  {bar}")

## 3. HyDE — Hypothetical Document Embeddings

In [ ]:
hyde_queries = [
    'I feel anxious about public speaking',
    'How do I cope with depression?',
    'I can\'t sleep because of stress',
]

print('HyDE vs plain embedding comparison:')
print(f'{"Query":<45}  Plain norm  HyDE norm  Same vector?')
print('-'*85)
for q in hyde_queries:
    plain = emb.embed_text(q)
    hyde  = emb.embed_hyde(q, alpha=0.5)
    sim_to_plain = emb.cosine_similarity(plain, hyde)
    # With no API key, HyDE == plain (fallback)
    same = abs(sim_to_plain - 1.0) < 1e-4
    print(f"  {q:<43}  {np.linalg.norm(plain):.4f}      {np.linalg.norm(hyde):.4f}     "
          f"{'yes (no key)' if same else f'no, sim={sim_to_plain:.3f}'}")

print()
if not api_key:
    print("ℹ  HyDE falls back to plain embedding when GROQ_API_KEY is not set.")
    print("   With a key, a hypothetical counsellor response is generated and")
    print("   blended 50/50 with the query embedding for richer retrieval.")
else:
    print("✓  GROQ_API_KEY set — HyDE generates real hypothetical documents.")

In [ ]:
# HyDE architecture diagram (text)
print("""
HyDE Pipeline
─────────────────────────────────────────────────────────
User query: "I feel anxious about public speaking"
     │
     ├─► embed_text(query)      ─────────────────► q_vec (384-d)
     │                                                    │
     └─► Groq LLM generates:                             │   alpha=0.5
         "Public speaking anxiety is common.              │
          Breathing techniques, visualisation             ▼
          and exposure therapy help…"           combined_vec = 0.5*q + 0.5*h
               │                                         │
               └─► embed_text(hypothetical) ─► h_vec    │
                                                         ▼
                                              L2-normalise → final embedding
─────────────────────────────────────────────────────────
Without key: combined_vec = q_vec  (passthrough)
""")

## 4. Query Optimizer

In [ ]:
opt = QueryOptimizer()
print(opt)

OPTIMIZE_TESTS = [
    "i feel anxious at work",
    "cant sleep feel hopeless",
    "my relationship is falling apart",
    "so stressed about money",
    "having panic attacks at school",
    "i think i might be depressed",
    "trouble concentrating and always tired",
]

opt_results = [opt.optimize(t) for t in OPTIMIZE_TESTS]

print(f'\n{"Original":<38}  {"Optimized":<65}  Method')
print('-'*115)
for r in opt_results:
    print(f"  {r['original']:<36}  {r['optimized'][:63]:<65}  {r['method']}")

In [ ]:
# Show token count expansion
print('Token expansion (synonym_expansion mode):')
print(f'{"Query":<38}  {"Original words":<16}  {"Expanded words":<16}  Expansion ratio')
print('-'*90)
for r in opt_results:
    orig_n = len(r['original'].split())
    optm_n = len(r['optimized'].split())
    ratio  = optm_n / orig_n if orig_n else 1
    print(f"  {r['original']:<36}  {orig_n:<16}  {optm_n:<16}  {ratio:.1f}x")

## 5. BM25 Search

In [ ]:
# Build a small knowledge base from representative chunks
KB_CHUNKS = [
    {"id":"kb_0","text":"Anxiety disorders involve persistent worry, fear and nervousness. Common treatments include CBT and medication.","source":"counseling"},
    {"id":"kb_1","text":"Depression is characterised by persistent low mood, loss of interest, fatigue and feelings of worthlessness.","source":"counseling"},
    {"id":"kb_2","text":"Cognitive Behavioral Therapy (CBT) is effective for anxiety, depression and panic disorder. It targets unhelpful thinking patterns.","source":"pdf"},
    {"id":"kb_3","text":"Insomnia and sleep disorders often accompany anxiety and depression. Sleep hygiene and relaxation techniques help.","source":"counseling"},
    {"id":"kb_4","text":"Mindfulness meditation and breathing exercises reduce stress and improve emotional regulation and wellbeing.","source":"pdf"},
    {"id":"kb_5","text":"Panic attacks involve sudden intense fear, racing heartbeat, shortness of breath and dizziness. They are treatable.","source":"counseling"},
    {"id":"kb_6","text":"Work-related stress and occupational burnout are common triggers for anxiety and depression in adults.","source":"pdf"},
    {"id":"kb_7","text":"Relationship difficulties, conflict and breakups are significant stressors that can lead to grief and depression.","source":"counseling"},
    {"id":"kb_8","text":"Self-harm and suicidal thoughts require immediate professional support. Crisis helplines are available 24/7.","source":"counseling"},
    {"id":"kb_9","text":"Trauma and PTSD symptoms include flashbacks, hypervigilance and emotional numbness. Trauma-focused therapy helps.","source":"pdf"},
]

hs = HybridSearch(KB_CHUNKS, alpha=0.5)
print(hs)

bm25_query = "anxiety and panic attacks"
bm25_results = hs.bm25_search(bm25_query, k=5)
print(f'\nBM25 results for: {bm25_query!r}')
print(f'{"Rank":<5}  {"ID":<7}  {"Score":<7}  Text')
print('-'*80)
for rank, (idx, score) in enumerate(bm25_results, 1):
    print(f"  {rank:<4}  {KB_CHUNKS[idx]['id']:<7}  {score:.3f}    {KB_CHUNKS[idx]['text'][:60]}")

## 6. Hybrid Search (BM25 + Semantic)

In [ ]:
# Pre-compute corpus vectors
hs.build_corpus_vectors(emb)
print('Corpus vectors built.')

SEARCH_QUERIES = [
    "I feel anxious about work and can't sleep",
    "How do I deal with panic attacks?",
    "I think I'm depressed and losing interest in everything",
    "My relationship broke down and I feel hopeless",
    "I'm having dark thoughts and don't know what to do",
]

for query in SEARCH_QUERIES:
    results = hs.search(query, emb, k=3)
    print(f'\n  Query: "{query}"')
    print(f'  {"Rank":<5}  {"ID":<7}  {"BM25":<6}  {"Sem":<6}  {"Comb":<6}  Chunk')
    print('  ' + '-'*80)
    for r in results:
        print(f"  {r['rank']:<5}  {r['chunk']['id']:<7}  {r['bm25_score']:.3f}  "
              f"{r['semantic_score']:.3f}  {r['combined_score']:.3f}  "
              f"{r['chunk']['text'][:55]}")

In [ ]:
# Alpha sensitivity — how weighting affects results
query = "anxiety coping strategies breathing exercises"
print(f'Alpha sensitivity for: "{query}"\n')
print(f'{"Alpha":<7}  {"Top-1 chunk ID":<12}  BM25   Sem    Comb   Text')
print('-'*90)
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r = hs.search(query, emb, k=1, alpha=alpha)[0]
    label = {0.0:'pure semantic',0.5:'equal hybrid',1.0:'pure BM25'}.get(alpha,'')
    print(f"  {alpha:<5}  {r['chunk']['id']:<12}  {r['bm25_score']:.3f}  "
          f"{r['semantic_score']:.3f}  {r['combined_score']:.3f}  "
          f"{r['chunk']['text'][:42]}  {label}")

In [ ]:
# Visualise BM25 vs semantic score comparison for one query
query = "I feel anxious and can't sleep"
results_full = hs.search(query, emb, k=len(KB_CHUNKS), alpha=0.5)

ids    = [r['chunk']['id'] for r in results_full]
bm25s  = [r['bm25_score']     for r in results_full]
sems   = [r['semantic_score'] for r in results_full]
combs  = [r['combined_score'] for r in results_full]

x = range(len(ids)); w = 0.26
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar([i-w   for i in x], bm25s, w, label='BM25',     color='#3498db')
ax.bar(list(x),             combs, w, label='Combined', color='#2ecc71')
ax.bar([i+w   for i in x], sems,  w, label='Semantic', color='#e74c3c')
ax.set_xticks(list(x)); ax.set_xticklabels(ids, rotation=30, ha='right')
ax.set_title(f'BM25 vs Semantic vs Hybrid scores\nQuery: "{query}"', fontsize=11)
ax.set_ylabel('Score (0-1)'); ax.set_ylim(0, 1.1)
ax.legend(); plt.tight_layout(); plt.show()

## 7. Full Pipeline Integration

In [ ]:
# Demonstrate full Phase 5 pipeline: NER → QueryOpt → HybridSearch
def rag_retrieve(user_text: str, top_k: int = 3) -> dict:
    """Run NER + query optimisation + hybrid search on a user message."""
    entities  = ner.extract(user_text)
    optimized = opt.optimize(user_text)
    results   = hs.search(optimized['optimized'], emb, k=top_k)
    return {
        'input':     user_text,
        'entities':  entities,
        'optimized': optimized['optimized'],
        'chunks':    results,
    }

DEMO_QUERIES = [
    "I've been feeling really anxious about work for the past few weeks",
    "I think I have panic attacks and can't sleep properly",
    "My relationship ended and I feel hopeless and depressed",
]

for q in DEMO_QUERIES:
    out = rag_retrieve(q)
    print(f"\n{'='*70}")
    print(f"  INPUT    : {out['input']}")
    print(f"  ENTITIES : symptoms={out['entities']['symptoms']}  "
          f"triggers={out['entities']['triggers']}  "
          f"sev={out['entities']['severity']}")
    print(f"  OPTIMIZED: {out['optimized'][:80]}")
    print(f"  TOP CHUNKS:")
    for r in out['chunks']:
        print(f"    [{r['rank']}] {r['chunk']['id']} comb={r['combined_score']:.3f}  "
              f"{r['chunk']['text'][:60]}")

In [ ]:
# Summary: Phase 5 components
print("""
Phase 5 — Component Summary
═══════════════════════════════════════════════════════════════════
 Component         File                          Mode (no key)
───────────────────────────────────────────────────────────────────
 NER Extractor     src/modules/ner_extractor.py  regex patterns
 Embedder          src/utils/embedder.py         local BERT model
 HyDE              src/utils/embedder.py         plain embed fallback
 Query Optimizer   src/utils/query_optimizer.py  synonym expansion
 BM25 Search       src/utils/hybrid_search.py    rank_bm25 (always)
 Hybrid Search     src/utils/hybrid_search.py    BM25+cosine
═══════════════════════════════════════════════════════════════════
 All components tested ✅  |  All degrade gracefully without API key
""")